In [1]:
import json
#import torch
from transformers import BertTokenizer, TFBertForSequenceClassification
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tqdm import tqdm

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import os
data_dir = '/content/drive/MyDrive/Healthcare_Sentiment/NewsAPI/train_items.jl'  # Replace with your actual path

In [ ]:
# Load data from JSON Lines file
#def load_data(file_path):
#    texts = []
#    labels = []
#    with open(file_path, 'r') as f:
#        for line in f:
#            st = r'%s' % line
#            data = json.loads(st)
#            texts.append(data['text'])
#            labels.append(data['label'])
#    return texts, labels

In [4]:
def load_data_with_cleaning(file_path):
        texts = []
        labels = []
        with open(file_path, 'r') as f:
            for line in f:
                try:
                    # Example cleaning: Replace literal backslashes with escaped ones
                    # This is a simplification and might not work for all issues!
                    cleaned_line = line.replace('\\"', '/')
                    #cleaned_line = cleaned_line1.replace('','')
                    # Ensure the cleaned line is a valid JSON string
                    data = json.loads(cleaned_line)
                    #combined_text = f"{data['title']} {data['text']}"
                    texts.append(data['title'])
                    #labels.append(data['label'])
                    # Convert label to integer
                    labels.append(int(data['label']))
                except json.JSONDecodeError as e:
                    print(f"Error decoding JSON on line: {e}")
                    print(f"Problematic line content (first 200 chars): {line[:1535]}")
                    # Decide how to handle errors (skip line, raise error, etc.)
                    continue # Skip the problematic line

        return texts, labels

In [5]:
texts, labels = load_data_with_cleaning(data_dir)

Error decoding JSON on line: Expecting property name enclosed in double quotes: line 1 column 1002 (char 1001)
Problematic line content (first 200 chars): {"source": {"id": null, "name": "The Providence Journal"}, "author": "Katie Mulvaney, Providence Journal", "title": "ICE provides details on arrest that sparked protest at Rhode Island Hospital. What we know", "description": "According to ICE, the man is a Dominican national who was previously arrested on domestic violence charges.", "url": "https://www.providencejournal.com/story/news/politics/courts/2025/04/26/ice-provides-details-on-mans-detention-that-led-to-april-24-providence-protest/83294897007/", "urlToImage": "https://s.yimg.com/ny/api/res/1.2/sDxe128Nbb9fYxBBJcLPVw--/YXBwaWQ9aGlnaGxhbmRlcjt3PTEyMDA7aD02NzU-/https://media.zenfs.com/en/the-providence-journal/85b8bd372d6d0e949947fba5d95dc06a", "publishedAt": "2025-04-26T17:49:47Z", "content": "PROVIDENCE Immigration and Customs Enforcement officials identified the man whose de

In [6]:
print(texts)

['Injectable Male Birth Control Effective for at Least 2 Years, Says Biotech Startup', "18 high-paying healthcare jobs that don't need a bachelor's degree", 'Semaglutide Shows Major Promise for Treating Serious Liver Disease', 'To See Within: Detecting X-Rays', 'Breaking the ‘intellectual bottleneck’: How AI is computing the previously uncomputible in healthcare', 'Immunotherapy drug capable of eliminating tumors in some early-stage cancers: Study', 'America’s Science Agency Says It Will Cut Funding to Researchers Who Protest Israel', "Newcastle's Howe back at work after hospital stay", "Howe is 'OK' but 'not 100%' after hospital stay", "Here's an exclusive look at the pitch deck that got an ex-Amazon exec $10 million to bring AI agents to health systems", 'Injectable Male Birth Control Effective for at Least 2 Years, Says Biotech Startup', "18 high-paying healthcare jobs that don't need a bachelor's degree", 'Semaglutide Shows Major Promise for Treating Serious Liver Disease', 'To See

In [ ]:
#load_data(data_dir)

In [8]:
#Split data
train_texts, val_texts, train_labels, val_labels = train_test_split(texts, labels, test_size=0.2, random_state=42)


In [9]:
print(f"Number of training samples: {len(train_texts)}")

Number of training samples: 54


In [10]:
# Load tokenizer and model
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
model = TFBertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=len(set(labels)))

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

All PyTorch model weights were used when initializing TFBertForSequenceClassification.

Some weights or buffers of the TF 2.0 model TFBertForSequenceClassification were not initialized from the PyTorch model and are newly initialized: ['classifier.weight', 'classifier.bias']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [11]:
# Tokenize data
train_encodings = tokenizer(list(train_texts), truncation=True, padding=True, max_length=100)
val_encodings = tokenizer(list(val_texts), truncation=True, padding=True, max_length=100)

In [12]:
# Convert to TensorFlow datasets
train_dataset = tf.data.Dataset.from_tensor_slices((
    dict(train_encodings),
    train_labels
)).batch(4)

In [13]:
val_dataset = tf.data.Dataset.from_tensor_slices((
    dict(val_encodings),
    val_labels
)).batch(4)

In [14]:
# Optimizer and loss
optimizer = tf.keras.optimizers.Adam(learning_rate=5e-5)
loss = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)
metric = tf.keras.metrics.SparseCategoricalAccuracy('accuracy')

In [15]:
# Compile model
model.compile(optimizer=optimizer, loss=loss, metrics=[metric])

In [16]:
# Train model
model.fit(train_dataset, epochs=5, validation_data=val_dataset
    )

Epoch 1/5
14/14 [==============================] - 51s 448ms/step - loss: 0.6812 - accuracy: 0.5926 - val_loss: 0.6212 - val_accuracy: 0.7143
Epoch 2/5
14/14 [==============================] - 1s 81ms/step - loss: 0.6844 - accuracy: 0.6296 - val_loss: 0.5614 - val_accuracy: 0.7857
Epoch 3/5
14/14 [==============================] - 1s 79ms/step - loss: 0.4720 - accuracy: 0.8519 - val_loss: 0.4367 - val_accuracy: 0.7857
Epoch 4/5
14/14 [==============================] - 1s 77ms/step - loss: 0.1623 - accuracy: 0.9630 - val_loss: 0.4430 - val_accuracy: 0.8571
Epoch 5/5
14/14 [==============================] - 1s 78ms/step - loss: 0.0356 - accuracy: 0.9815 - val_loss: 0.6276 - val_accuracy: 0.7857


In [17]:
# Evaluate model
loss, accuracy = model.evaluate(val_dataset)
print(f"Loss: {loss}, Accuracy: {accuracy}")

4/4 [==============================] - 0s 25ms/step - loss: 0.6276 - accuracy: 0.7857
Loss: 0.6275848150253296, Accuracy: 0.7857142686843872


In [18]:
# Make predictions
text = "This movie was great!"
predict_input = tokenizer(text, truncation=True, padding=True, return_tensors='tf')
output = model(predict_input)[0]
prediction_value = tf.argmax(output, axis=1).numpy()[0]
print(f"Predicted sentiment: {prediction_value}")

Predicted sentiment: 1


In [19]:
#File to Apply model to
new_data_dir = '/content/drive/MyDrive/Healthcare_Sentiment/NewsAPI/items.jl'  # Replace with your actual path

In [23]:
# Function to load data from a JSON Lines file (similar to your existing function)
def load_new_data_with_cleaning(file_path):
    contents = []
    texts = []
    ids = [] # Assuming each item has a unique ID you want to keep

    source = []
    descriptions = []
    author = []
    timestamp = []
    urlImage = []
    tags = []

    with open(file_path, 'r') as f:
        for line in f:
            try:
                cleaned_line = line.replace('\\"', '/')
                data = json.loads(cleaned_line)
                # Assuming you want to predict on the 'title' again
                texts.append(data['title'])
                contents.append(data['content'])


                source.append(data['source'])
                descriptions.append(data['description'])
                author.append(data['author'])
                timestamp.append(data['publishedAt'])
                urlImage.append(data['urlToImage'])
                tags.append(data['tags'])


                # Assuming an 'id' field exists to track the original item
                if 'url' in data:
                  ids.append(data['url'])
                else:
                  ids.append(None) # Or handle cases without an ID
            except json.JSONDecodeError as e:
                print(f"Error decoding JSON on line: {e}")
                print(f"Problematic line content (first 200 chars): {line[:1535]}")
                continue
    return contents, texts, ids, source, descriptions, author, timestamp, urlImage, tags

# Load the new data
new_content, new_texts, new_ids, new_source, new_description, new_author, new_timestamp, new_urlToImage, new_tags = load_new_data_with_cleaning(new_data_dir)

# Tokenize the new data
new_encodings = tokenizer(list(new_texts), truncation=True, padding=True, max_length=100, return_tensors='tf')

# Create a TensorFlow dataset for the new data
new_dataset = tf.data.Dataset.from_tensor_slices(
    dict(new_encodings)
).batch(4)

# Make predictions
predictions = model.predict(new_dataset)

# Get the predicted class (index with the highest probability)
predicted_labels = tf.argmax(predictions.logits, axis=1).numpy()

# Write the results back to a new file or overwrite the original
output_data_dir = '/content/drive/MyDrive/Healthcare_Sentiment/CNBC/new_items_with_predictions.jl' # Define output file path

with open(output_data_dir, 'w') as outfile:
    for i, text in enumerate(new_texts):
        result = {
            'source': new_source[i],
            'author': new_author[i],
            'title': new_texts[i],
            'description': new_description[i],
            'url': new_ids[i], # Include the original ID if available
            'urlToImage': new_urlToImage[i],
            'publishedAt': new_timestamp[i],
            'content': new_content[i],
            'tags': new_tags[i],
            'predicted_label': int(predicted_labels[i]) # Ensure it's an integer
        }
        outfile.write(json.dumps(result) + '\n')

print(f"Predictions written to {output_data_dir}")

246/246 [==============================] - 8s 33ms/step
Predictions written to /content/drive/MyDrive/Healthcare_Sentiment/CNBC/new_items_with_predictions.jl
